# 02d - 从 TRIZ-raw corpus 生成 SFT 微调数据集

本 notebook 读取 `data/processed/corpus/triz_corpus.jsonl`，调用 Moonshot API 为每个文本片段生成高质量的 instruction/output 问答对，
并映射到项目定义的 6 个数据子集。最后转换为 ChatML 格式并保存 train/validation/test 分割。

> 注意：调用 Moonshot API 需要设置环境变量 `MOONSHOT_API_KEY`，并会产生 API 费用。

In [ ]:
import sys
sys.path.append('/home/meerkat/mongoose_ai')

from config import CORPUS_CONFIG, DATA_CONFIG
from utils.corpus_to_sft import CorpusSFTGenerator

corpus_path = CORPUS_CONFIG['output_dir'] + '/' + CORPUS_CONFIG['output_filename']

# 生成参数
MAX_SAMPLES = 500       # 设为 None 表示处理全部 corpus chunks
BATCH_SIZE = 5          # 每个 API 请求包含的 chunk 数
RPM = 3                 # Moonshot Tier 0 默认限制

print(f"corpus 路径: {corpus_path}")

## 1. 估算成本

In [ ]:
import json

# 统计 corpus 总 chunk 数
total_chunks = sum(1 for _ in open(corpus_path, 'r', encoding='utf-8'))
sample_count = MAX_SAMPLES if MAX_SAMPLES is not None else total_chunks

estimate = CorpusSFTGenerator.estimate_cost(sample_count, batch_size=BATCH_SIZE, rpm=RPM)
print(json.dumps(estimate, ensure_ascii=False, indent=2))

## 2. 生成 SFT 样本

支持断点续跑：如果中途中断，重新运行会从检查点自动恢复。

In [ ]:
generator = CorpusSFTGenerator(
    model="moonshot-v1-8k",
    rpm=RPM,
    output_dir="/home/meerkat/mongoose_ai/data/raw/corpus_sft",
    checkpoint_dir="/home/meerkat/mongoose_ai/data/processed/checkpoint_corpus_sft",
)

grouped, stats = generator.generate_from_corpus(
    corpus_path=corpus_path,
    max_samples=MAX_SAMPLES,
    batch_size=BATCH_SIZE,
    temperature=0.7,
)

print("\n生成统计:")
print(json.dumps(stats, ensure_ascii=False, indent=2))

In [ ]:
# 保存为 6 个子集 JSON 文件
saved_files = generator.save_subsets(grouped)
print("\n已保存子集文件:")
for f in saved_files:
    print(f"  {f}")

## 3. 查看生成样本示例

In [ ]:
for subset_name, samples in grouped.items():
    if samples:
        print(f"\n=== {subset_name} ({len(samples)} 条) ===")
        s = samples[0]
        print("instruction:", s['instruction'][:200])
        print("output:", s['output'][:300] + "...")
        break

## 4. 转换为 ChatML 格式并划分数据集

In [ ]:
from utils.data_utils import load_raw_data, convert_to_chatml, save_dataset
from utils.training_utils import load_model_and_tokenizer
from config import MODELS_DIR, BASE_MODEL, DATA_CONFIG
import os

# 加载生成的 SFT 子集
raw_data = load_raw_data("/home/meerkat/mongoose_ai/data/raw/corpus_sft")

print(f"已加载 {len(raw_data)} 个子集:")
for name, samples in raw_data.items():
    print(f"  {name}: {len(samples)} 条")

In [ ]:
# 加载 tokenizer 以使用官方 chat template
model_path = os.path.join(MODELS_DIR, BASE_MODEL.split('/')[-1])
_, tokenizer = load_model_and_tokenizer(
    model_name_or_path=model_path,
    quantization_config=None,
    device_map='cpu',
)

dataset = convert_to_chatml(
    data=raw_data,
    tokenizer=tokenizer,
    system_message=DATA_CONFIG['chatml']['system_message']
)

print("数据集划分:")
for split_name, split_data in dataset.items():
    avg_len = sum(len(s['text']) for s in split_data) / len(split_data)
    print(f"  {split_name}: {len(split_data)} 条，平均 {avg_len:.0f} 字符")

## 5. 保存最终数据集

In [ ]:
# 保存到 data/processed/（默认路径，Notebook 04 会读取）
# 如果你想保留原始 sample_data 生成的 processed 数据，请修改 output_dir
processed_dir = DATA_CONFIG['processed_data_dir']
save_dataset(dataset, processed_dir)

print(f"数据集已保存到: {processed_dir}")
import os
for f in sorted(os.listdir(processed_dir)):
    if f.endswith('.jsonl'):
        print(f"  {f}")

## 下一步

数据集准备完成后，打开 `03_model_benchmark.ipynb` 进行基准评测，或直接运行 `04_qlora_finetune.ipynb` 开始微调。